# 📊 Asian Options & Pandas Integration
Asian options are path-dependent. Their payoff depends on the arithmetic average of the asset's trajectory. This requires simulating the SDE step-by-step. 

* We will use `m_steps = 252` (daily observations over 1 year).
* Let's run a batch of 10 simulations to build a Volatility Surface.
* **Cost per Simulation:** N=100,000 × M=252 = 25.2 Million Steps = **~0.10 Credits**.
* Total batch cost: ~1.00 Credit.

In [ ]:
import requests
import uuid
import pandas as pd
import time

API_KEY = "pmt_live_your_secure_api_key_here"
BASE_URL = "https://api.prometheusquantengine.com/api/v1/simulations"

volatilities = [0.15, 0.20, 0.25, 0.30, 0.35]
strikes = [90.0, 100.0]

results = []

print("Igniting C++ OpenMP cores. Dispatching 10 synchronous matrices...\n")
start_batch = time.time()

for vol in volatilities:
    for k in strikes:
        payload = {
            "simulation_type": "Asian",
            "s_0": 100.0,
            "strike": k,
            "volatility": vol,
            "time_to_maturity": 1.0,
            "risk_free_rate": 0.05,
            "option_type": "Call",
            "n_simulations": 100000,
            "m_steps": 252
        }
        
        headers = {"X-API-Key": API_KEY, "Idempotency-Key": str(uuid.uuid4())}
        resp = requests.post(BASE_URL, json=payload, headers=headers).json()
        
        results.append({
            "Strike": k,
            "Volatility": f"{vol*100}%",
            "Fair Value": round(resp.get('fair_value', 0), 4),
            "Gamma (Γ)": round(resp.get('gamma', 0), 6),
            "Compute Latency (s)": round(time.time() - start_batch, 4)
        })

df = pd.DataFrame(results)
print(f"Batch completed in {time.time() - start_batch:.2f} seconds.")
display(df) # Renders a beautiful table in Colab